In [84]:
import folium
import pandas as pd

In [85]:
from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon

In [86]:
df = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"
)
df.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


In [87]:
df.columns

Index(['Flight Number', 'Date', 'Time (UTC)', 'Booster Version', 'Launch Site',
       'Payload', 'Payload Mass (kg)', 'Orbit', 'Customer', 'Landing Outcome',
       'class', 'Lat', 'Long'],
      dtype='object')

In [88]:
df = df[["Launch Site", "Lat", "Long", "class"]]
launch_sites_df = df.groupby(["Launch Site"], as_index=False).first()
launch_sites_df = launch_sites_df[["Launch Site", "Lat", "Long"]]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [89]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [90]:
circle = folium.Circle(
    nasa_coordinate, radius=1000, color="#d35400", fill=True
).add_child(folium.Popup("NASA Johnson Space Center"))
marker = folium.map.Marker(
    nasa_coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "NASA JSC",
    ),
)
site_map.add_child(circle)
site_map.add_child(marker)
site_map.save("map1_nasa.html")

In [91]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for index, site in launch_sites_df.iterrows():
    coordinate = [site["Lat"], site["Long"]]

    circle = folium.Circle(
        coordinate, radius=1000, color="#d35400", fill=True
    ).add_child(folium.Popup(site["Launch Site"]))

    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>'
            % site["Launch Site"],
        ),
    )

    site_map.add_child(circle)
    site_map.add_child(marker)

site_map.save("map.html")
site_map.save("map2_launch_sites.html")

In [92]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for index, record in df.iterrows():
    folium.Marker(
        [record["Lat"], record["Long"]],
        icon=folium.Icon(color="green" if record["class"] == 1 else "red"),
        popup=folium.Popup(
            f"Site: {record['Launch Site']} | Outcome: {'Success' if record['class']==1 else 'Failure'}",
            max_width=200,
        ),
    ).add_to(site_map)

site_map.save("map3_success_failure.html")

In [93]:
df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


In [94]:
df["marker_color"] = df["class"].apply(lambda x: "green" if x == 1 else "red")
df.head()

,Launch Site,Lat,Long,class,marker_color
0,CCAFS LC-40,28.562302,-80.577356,0,red
1,CCAFS LC-40,28.562302,-80.577356,0,red
2,CCAFS LC-40,28.562302,-80.577356,0,red
3,CCAFS LC-40,28.562302,-80.577356,0,red
4,CCAFS LC-40,28.562302,-80.577356,0,red


In [95]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)
for index, record in df.iterrows():
    marker = folium.Marker(
        [record["Lat"], record["Long"]],
        icon=folium.Icon(color="white", icon_color=record["marker_color"]),
        popup=folium.Popup(
            f"Site: {record['Launch Site']} | Outcome: {'Success' if record['class']==1 else 'Failure'}",
            max_width=200,
        ),
    )
    marker_cluster.add_child(marker)

site_map.save("map4_clusters.html")

In [96]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position="topright",
    separator=" Long: ",
    empty_string="NaN",
    lng_first=False,
    num_digits=20,
    prefix="Lat:",
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map.save("map5_proximity.html")

In [97]:
from math import sin, cos, sqrt, atan2, radians


def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [98]:
launch_site_lat = 28.562302
launch_site_lon = -80.577356
coastline_lat = 28.56367
coastline_lon = -80.57163
distance_coastline = calculate_distance(
    launch_site_lat, launch_site_lon, coastline_lat, coastline_lon
)
print(f"Distance to coastline: {distance_coastline:.2f} KM")
coordinate = [coastline_lat, coastline_lon]
distance_marker = folium.Marker(
    coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>'
        % "{:10.2f} KM".format(distance_coastline),
    ),
)
site_map.add_child(distance_marker)
lines = folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], coordinate], weight=1
)
site_map.add_child(lines)
site_map.save("map6_distances.html")

Distance to coastline: 0.58 KM


In [99]:
city_lat = 28.3922
city_lon = -80.6077
distance_city = calculate_distance(launch_site_lat, launch_site_lon, city_lat, city_lon)

city_marker = folium.Marker(
    [city_lat, city_lon],
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#0000ff;"><b>%s</b></div>'
        % "{:10.2f} KM".format(distance_city),
    ),
)
site_map.add_child(city_marker)
site_map.add_child(
    folium.PolyLine(
        locations=[[launch_site_lat, launch_site_lon], [city_lat, city_lon]], weight=1
    )
)

railway_lat = 28.56312
railway_lon = -80.58697
distance_railway = calculate_distance(
    launch_site_lat, launch_site_lon, railway_lat, railway_lon
)

railway_marker = folium.Marker(
    [railway_lat, railway_lon],
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#00aa00;"><b>%s</b></div>'
        % "{:10.2f} KM".format(distance_railway),
    ),
)
site_map.add_child(railway_marker)
site_map.add_child(
    folium.PolyLine(
        locations=[[launch_site_lat, launch_site_lon], [railway_lat, railway_lon]],
        weight=1,
    )
)

highway_lat = 28.56386
highway_lon = -80.57086
distance_highway = calculate_distance(
    launch_site_lat, launch_site_lon, highway_lat, highway_lon
)

highway_marker = folium.Marker(
    [highway_lat, highway_lon],
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#ff0000;"><b>%s</b></div>'
        % "{:10.2f} KM".format(distance_highway),
    ),
)
site_map.add_child(highway_marker)
site_map.add_child(
    folium.PolyLine(
        locations=[[launch_site_lat, launch_site_lon], [highway_lat, highway_lon]],
        weight=1,
    )
)

site_map.save("map6_distances.html")